In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END, START
from typing import Annotated
from langgraph.graph.message import add_messages

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=GROQ_API_KEY)

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

def superbot(state: State):
    return {"messages" : [llm.invoke(state['messages'])]}

In [6]:
graph_builder = StateGraph(State)

graph_builder.add_node("superbot", superbot)

graph_builder.add_edge(START, "superbot")

graph_builder.add_edge("superbot", END)

In [7]:
graph = graph_builder.compile(checkpointer=memory)

In [8]:
config = {
    "configurable" : {
        "thread_id" : "2"
    }
} 

In [9]:
for chunk in graph.stream({"messages" : "Hi, my name is Prashant and I like Cricket"}, config=config, stream_mode="updates"):
    print(chunk)


{'superbot': {'messages': [AIMessage(content='Hey Prashant! 👋 It’s great to meet a cricket fan. Are you a bowler, a batsman, or just a big-picture enthusiast? What’s your favorite team or player? Let me know—I’d love to chat about the game!', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Prashant and I like Cricket". The assistant should respond in a friendly manner, acknowledging that. Also we could ask about cricket. The user might want to talk about cricket. So respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 83, 'total_tokens': 195, 'completion_time': 0.122056544, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.005524018, 'prompt_tokens_details': None, 'queue_time': 0.050314632, 'total_time': 0.127580562}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='

In [10]:
for chunk in graph.stream({"messages" : "Hi, my name is Prashant and I like Cricket"}, config=config, stream_mode="values"):
    print(chunk)


{'messages': [HumanMessage(content='Hi, my name is Prashant and I like Cricket', additional_kwargs={}, response_metadata={}, id='db20f9ec-21db-4df2-a6b8-30e396c49403'), AIMessage(content='Hey Prashant! 👋 It’s great to meet a cricket fan. Are you a bowler, a batsman, or just a big-picture enthusiast? What’s your favorite team or player? Let me know—I’d love to chat about the game!', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Prashant and I like Cricket". The assistant should respond in a friendly manner, acknowledging that. Also we could ask about cricket. The user might want to talk about cricket. So respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 83, 'total_tokens': 195, 'completion_time': 0.122056544, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.005524018, 'prompt_tokens_details': None, 'queue_time': 0.050314632, 'total_time': 0.127580562}, 'model_name': 'openai/gpt-oss-20b', 

In [11]:
for chunk in graph.stream({"messages" : "I also like football"}, config=config, stream_mode="updates"):
    print(chunk)


{'superbot': {'messages': [AIMessage(content='That’s awesome—so you’re a double‑sport fan! 🎉\n\n- **Football**: Are you into the Premier League, La Liga, Serie A, or maybe the World Cup? Do you have a favorite club or player?  \n- **Cricket**: Which format gets your heart racing? Tests, ODIs, T20s, or a franchise league like the IPL/BBL?  \n- **Cross‑overs**: Any athletes you admire who play both sports or have a strong presence in both?\n\nLet me know which side you’re leaning toward today, or if you want a mix of both—happy to share recent highlights, stats, or even some trivia!', additional_kwargs={'reasoning_content': "We need to respond, continuing conversation. It's a casual chat. We should ask about preferences. Provide content."}, response_metadata={'token_usage': {'completion_tokens': 172, 'prompt_tokens': 367, 'total_tokens': 539, 'completion_time': 0.17978109, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.023204159, 'prompt_tokens_details': None, 'q

In [12]:
for chunk in graph.stream({"messages" : "I also like football"}, config=config, stream_mode="values"):
    print(chunk)


{'messages': [HumanMessage(content='Hi, my name is Prashant and I like Cricket', additional_kwargs={}, response_metadata={}, id='db20f9ec-21db-4df2-a6b8-30e396c49403'), AIMessage(content='Hey Prashant! 👋 It’s great to meet a cricket fan. Are you a bowler, a batsman, or just a big-picture enthusiast? What’s your favorite team or player? Let me know—I’d love to chat about the game!', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Prashant and I like Cricket". The assistant should respond in a friendly manner, acknowledging that. Also we could ask about cricket. The user might want to talk about cricket. So respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 83, 'total_tokens': 195, 'completion_time': 0.122056544, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.005524018, 'prompt_tokens_details': None, 'queue_time': 0.050314632, 'total_time': 0.127580562}, 'model_name': 'openai/gpt-oss-20b', 

### Asnyc Streaming

In [13]:
config = {
    "configurable" : {
        "thread_id" : "3"
    }
} 

async for event in graph.astream_events({"messages" : "Hi, my name is Prashant and I like Cricket"}, config=config, version="v2"):
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': 'Hi, my name is Prashant and I like Cricket'}}, 'name': 'LangGraph', 'tags': [], 'run_id': '019e01b0-fbc5-7641-978c-5b9f87fde919', 'metadata': {'thread_id': '3', 'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='Hi, my name is Prashant and I like Cricket', additional_kwargs={}, response_metadata={}, id='0bd86dd1-16a9-4607-a8e4-1a0731790c22')]}}, 'name': 'superbot', 'tags': ['graph:step:1'], 'run_id': '019e01b0-fbc6-76a1-8db8-214d74480c8c', 'metadata': {'thread_id': '3', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'superbot', 'langgraph_triggers': ('branch:to:superbot',), 'langgraph_path': ('__pregel_pull', 'superbot'), 'langgraph_checkpoint_ns': 'superbot:f30af335-2cd0-1157-df1d-293845c2ecb7'}, 'parent_ids': ['019e01b0-fbc5-7641-978c-5b9f87fde919']}
{'event': 'on_chat_model_start', 'data': {'input': {'messages': [[HumanMe